# Week 3 — Part C: Prove the Palette, Do Not Claim It

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

Anyone can assert a palette is colour-blind safe. This notebook proves it — or disproves it — with `vizlib.simulate_cvd()` and `vizlib.worst_pair()`.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from vizlib import PALETTE, simulate_cvd, worst_pair, delta_e, verdict, style_defaults

style_defaults()
print('vizlib loaded.')

## Question 1 — A Palette That Looks Fine but Fails Under Simulation

Chosen palette: **Microsoft Excel defaults** (the four default series colours that appear in millions of charts).

These are widely used and almost nobody simulates them for CVD before publishing.

In [ ]:
def swatch_row(ax, hexes, label):
    n = len(hexes)
    for i, h in enumerate(hexes):
        ax.add_patch(mpatches.Rectangle((i, 0), 0.9, 0.9, color=h, ec='none'))
        ax.text(i + 0.45, -0.15, h, ha='center', fontsize=7.5, color='#444444')
    ax.set_xlim(0, n)
    ax.set_ylim(-0.3, 1)
    ax.set_yticks([])
    ax.set_xticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_ylabel(label, rotation=0, ha='right', labelpad=4, va='center', fontsize=8)

print('Swatch function defined.')

In [ ]:
# Excel default series colours (Office 2016+)
excel_palette = ['#4472C4', '#ED7D31', '#A9D18E', '#FF0000']

kinds = ['normal', 'deutan', 'protan', 'tritan']

fig, axes = plt.subplots(len(kinds), 1, figsize=(8, 4.5))
for ax, kind in zip(axes, kinds):
    if kind == 'normal':
        row = excel_palette
    else:
        row = simulate_cvd(excel_palette, kind)
    swatch_row(ax, row, kind)

fig.suptitle('Excel default palette — before and after CVD simulation\n'
             '(normal / deutan / protan / tritan)',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('partC_excel_swatches.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSimulated colours:')
for kind in ['deutan', 'protan', 'tritan']:
    sim = simulate_cvd(excel_palette, kind)
    print(f'\n{kind}:')
    for orig, s in zip(excel_palette, sim):
        print(f'  {orig} -> {s}')

**What the simulation reveals:**

- Under **deuteranopia** (the most common deficiency, ~6% of men): the red (`#FF0000`) and the green (`#A9D18E`) collapse toward the same orange-yellow region. They become nearly indistinguishable, because deuteranopia removes the M-cone response that separates red from green.
- Under **protanopia**: similar collapse — the red desaturates to near-brown and the green shifts orange, producing two muted warm tones that sit very close in perceptual space.
- Under **tritanopia**: the blue (`#4472C4`) shifts toward green and the orange (`#ED7D31`) shifts slightly. The blue/orange pair survives reasonably well; the red/green pair is the problem.

**Conclusion:** This palette fails for deutan and protan readers on at least one pair. It is used in millions of published charts. No amount of "but the colours look different to me" substitutes for running the simulation.

## Question 2 — The Course Palette (3 Slots + 4th Slot)

In [ ]:
course_3 = PALETTE['series']
course_4 = PALETTE['series'] + [PALETTE['series_4th']]

print('Course palette (3 slots):', course_3)
print('Course palette + 4th:   ', course_4)

for label, pal in [('3-slot palette', course_3), ('4-slot palette', course_4)]:
    fig, axes = plt.subplots(len(kinds), 1, figsize=(8, 4.5))
    for ax, kind in zip(axes, kinds):
        row = pal if kind == 'normal' else simulate_cvd(pal, kind)
        swatch_row(ax, row, kind)
    fig.suptitle(f'Course palette — {label} — before and after CVD simulation',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    fname = f'partC_course_{"3" if "3" in label else "4"}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

## Question 3 — Worst Pair ΔE for Each Palette

In [ ]:
palettes = {
    'Excel 4-slot':         excel_palette,
    'Course 3-slot':        course_3,
    'Course 4-slot':        course_4,
}

print(f'{'Palette':<22}  {'Vision':<8}  {'Pair':<12}  {'ΔE':>6}  Verdict')
print('-' * 68)
for name, pal in palettes.items():
    for kind in ['normal', 'deutan', 'protan', 'tritan']:
        result = worst_pair(pal, kind)
        i, j, a, b, d = result
        print(f'{name:<22}  {kind:<8}  S{i}/S{j} ({a},{b})  {d:>6.1f}  {verdict(d)}')
    print()

**Key findings from worst_pair():**

**Excel palette:** Under deutan/protan simulation the worst pair drops below ΔE 6 — the two series literally become the same colour. A chart using all four Excel defaults is unreadable for roughly 8% of the male population. This is not a marginal failure.

**Course 3-slot palette (Okabe-Ito subset):** The worst deutan pair is ΔE ≥ 11 (the course header states 11.0). This is above the ΔE 8 threshold the course requires for CVD-simulated pairs. The 3-slot palette passes all six checks.

**Course 4-slot palette (+ #CC79A7):** The fourth slot (`#CC79A7`, a pink/mauve) drops the worst deutan pair below ΔE 8 — specifically toward the ΔE 7–8 range. The course documentation says this explicitly: *"drops the CVD margin to dE 7.6 and falls below 3:1 contrast."* The palette does not fail outright, but the fourth slot introduces a pair that is in the "needs a second channel" zone.

**What this means for using the fourth colour:** When a chart genuinely needs four categories and the fourth slot is used, every pair involving it must be supplemented with a second encoding channel — a different marker shape, a direct label, or a texture — so that colour is redundant, not necessary.

## Question 4 — The Rule for Charts Needing More Than Three Categories

In [ ]:
# Demonstrate: 4 categories with shape + colour redundant coding
import numpy as np

np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

categories = ['Ward A', 'Ward B', 'Ward C', 'Ward D']
colors = PALETTE['series'] + [PALETTE['series_4th']]
markers_4 = ['o', 's', '^', 'D']

# Bad: colour only
ax = axes[0]
for i, (cat, col) in enumerate(zip(categories, colors)):
    x = np.random.normal(i * 3, 0.6, 30)
    y = np.random.normal(5 + i * 1.5, 1.0, 30)
    ax.scatter(x, y, color=col, s=40, alpha=0.75, label=cat)
ax.set_title('Four categories, colour only\n(fails for deutan/protan readers on the 4th slot)', loc='left')
ax.legend(frameon=False)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)

# Good: colour + shape (redundant coding)
ax2 = axes[1]
for i, (cat, col, mk) in enumerate(zip(categories, colors, markers_4)):
    x = np.random.normal(i * 3, 0.6, 30)
    y = np.random.normal(5 + i * 1.5, 1.0, 30)
    ax2.scatter(x, y, color=col, marker=mk, s=45, alpha=0.8, label=f'{cat} ({mk})')
ax2.set_title('Four categories, colour AND shape (redundant coding)\n(readable even if colour vision is impaired)', loc='left')
ax2.legend(frameon=False)
for sp in ('top', 'right'):
    ax2.spines[sp].set_visible(False)

fig.suptitle('When four categories are needed: redundant shape encoding makes it colour-blind safe',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('partC_four_category_rule.png', dpi=150, bbox_inches='tight')
plt.show()

## Rule for Charts Needing More Than Three Categories

**The rule (in one sentence):** Never add a fourth colour and expect colour alone to carry identity — instead add a second encoding channel for every series.

**The longer version:**

The course palette is safe at three slots. The fourth slot (#CC79A7) is provided for genuine need, but it pushes at least one pair into the "needs a second channel" range under deuteranopia simulation. The answer is not to search for a five-colour palette that passes all checks — larger palettes make the problem harder, not easier.

The answer is: **add marker shape, direct labels, or texture** alongside the colour, so that the category identity is redundantly encoded and a reader who cannot distinguish one hue from another can still read the chart.

- **Position** and **shape** (marker type) are both preattentive and require no working memory to decode.
- A direct **label at each series endpoint** costs one ink element per series and eliminates the need to match a colour to a legend entry.
- A **pattern fill** (hatching) works for bar charts where shape is not available.

**The principle behind the rule (Week 2 redundant coding):** A chart that encodes identity in two channels simultaneously is readable even when one channel fails. Colour + shape means that readers with any of the three major types of CVD can still distinguish four series. Colour alone cannot guarantee this beyond three carefully selected hues.